**Name :** Sarvajit Patil

**ROll No.:** DS82

**PRN :** 2425010120

**GenAI Experiment 7**

In [ ]:
!pip install openai --quiet

In [ ]:
!pip install google-genai --quiet

In [ ]:
# Initialize Gemini client using Google GenAI SDK
from google import genai
from google.colab import userdata
from getpass import getpass
import os

# Try loading from Secrets first, otherwise prompt for input
try:
    api_key = userdata.get('GEMINI_API_KEY')
except Exception:
    api_key = os.environ.get('GEMINI_API_KEY')

if not api_key:
    api_key = getpass("Enter your Gemini API key: ")
    os.environ["GEMINI_API_KEY"] = api_key

client = genai.Client(api_key=api_key)
print("Gemini client initialized successfully.")

Enter your Gemini API key: ··········
Gemini client initialized successfully.


In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [ ]:
#Task 1 — Create the OpenAI client
import os
from openai import OpenAI

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY")
)

print("OpenAI client created successfully.")

OpenAI client created successfully.


In [ ]:
#Task 2 — Create conversation history
messages = [
    {
        "role": "system",
        "content": "You are a friendly and concise college lab assistant."
    }
]

print("Initial messages:", messages)

Initial messages: [{'role': 'system', 'content': 'You are a friendly and concise college lab assistant.'}]


In [ ]:
#Task 3 — Create the chat() function
def chat(user_input):

    # Add user's question to conversation history
    messages.append({
        "role": "user",
        "content": user_input
    })

    # Send the complete conversation history to the model
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages
    )

    # Get the assistant's reply
    reply = response.choices[0].message.content

    # Add assistant's reply to conversation history
    messages.append({
        "role": "assistant",
        "content": reply
    })

    return reply

In [ ]:
# Task 3 (Alternative) — Create the chat() function using Gemini Chat Session
from google.genai import types

# Initialize a native Gemini chat session with the system instruction
chat_session = client.chats.create(
    model="gemini-2.5-flash",
    config=types.GenerateContentConfig(
        system_instruction="You are a friendly and concise college lab assistant."
    )
)

def chat(user_input):
    # Send message to the chat session (it automatically appends history)
    response = chat_session.send_message(user_input)

    # Sync the messages list so your len(messages) prints still work correctly
    global messages
    messages = []
    # Add the system instruction at the beginning of the list
    messages.append({
        "role": "system",
        "content": "You are a friendly and concise college lab assistant."
    })
    # Get all history from the chat session and convert back to your messages list
    history = chat_session.get_history()
    for content in history:
        role = "assistant" if content.role == "model" else content.role
        text_content = "".join([part.text for part in content.parts if part.text])
        messages.append({
            "role": role,
            "content": text_content
        })

    return response.text

In [ ]:
#Task 4 — Run Question 1
answer1 = chat("What is supervised learning?")

print("Assistant:", answer1)
print("Number of messages:", len(messages))

Assistant: Supervised learning is a type of machine learning where you train a model using a dataset that already has the correct answers, or "labels," for each piece of input data.

Think of it like teaching a child: you show them pictures of cats and tell them, "This is a cat." The model learns from these examples to then predict those labels for new, unseen data.
Number of messages: 3


In [ ]:
#Run Question 2
answer2 = chat("Give one real-world example of it.")

print("Assistant:", answer2)
print("Number of messages:", len(messages))


Assistant: A common example is **email spam detection**.

You feed the model a huge dataset of emails, where each email is manually labeled as either "spam" or "not spam." The model learns the patterns that distinguish spam from legitimate emails, and then it can predict whether a new incoming email is spam or not.
Number of messages: 5


In [ ]:
#Run Question 3
answer3 = chat("Which of the models we discussed uses labels?")

print("Assistant:", answer3)
print("Number of messages:", len(messages))

Assistant: **Supervised learning** is the model we discussed that uses labels.

Specifically, in the example of email spam detection, the model uses labels like "spam" or "not spam."
Number of messages: 7


In [ ]:
# Observation

'''
The chatbot remembers the previous conversation because the complete messages history is sent with every API request.
Therefore, Question 2 can understand “it” as supervised learning, and Question 3 can identify the model that uses labels from the previous discussion.
Without history, these follow-up questions may not have enough context.

'''


In [ ]:
'''
Viva questions — important
Q1. What is a conversational chatbot?
A conversational chatbot is an application that uses an LLM to interact with users over multiple turns while maintaining conversation context.

Q2. What is conversation history?
Conversation history is the list containing the system message and all previous user and assistant messages.

Q3. Why do we maintain messages?
We maintain messages so that previous conversation context can be sent to the model with the next question.

Q4. Is the Chat API stateful?
No. The API is stateless. Each API call is processed independently. The application creates memory by sending the conversation history with every request.

Q5. What are the three message roles?
system
user
assistant
system defines the chatbot's behavior, user contains the user's question, and assistant contains the model's previous response.

Q6. What does response.choices[0].message.content do?
It extracts the text of the assistant's response from the API result.

Q7. Why append the assistant response to messages?
Because the next API call needs to know what the assistant previously said.

Q8. Why does the number of messages increase by two each turn?
Each turn adds:
1 user message
1 assistant message
The initial system message remains in the list.

Q9. What is a token?
A token is a unit of text processed by the model and used for usage/billing calculations.

Q10. What happens to cost as conversation history grows?
Because the entire history is resent on every turn, the input size grows. Long conversations can therefore increase cost and latency and can eventually approach the model's context-window limit.

Q11. How can long conversation history be controlled?
The reference gives four approaches:
1. Sliding window
2. Summarization
3. Token budget
4. External memory